**Programmer: python_scripts (Abhijith Warrier)**

**PYTHON SCRIPT TO *BUILD A SEMANTIC SEARCH ENGINE THAT RETRIEVES INFORMATION BASED ON MEANING, NOT JUST KEYWORDS. 🐍🔍***

Traditional search engines primarily rely on matching keywords. But what happens when a user asks a question using different words from those stored in a document?

**Semantic Search** solves this problem by comparing the meaning of text using vector embeddings.

In this project, we'll build a simple semantic search engine using **FAISS (Facebook AI Similarity Search)** and Sentence Transformers to retrieve relevant documents based on their semantic similarity. We'll cover document embeddings, vector indexing, cosine similarity, and retrieving the most relevant results.

---

## 📦 Step 1 — Install Required Libraries

We'll use Sentence Transformers to generate embeddings, FAISS for similarity search, and NumPy for numerical operations.

```python
pip install faiss-cpu sentence-transformers numpy
```

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

---

## 📚 Step 2 — Create a Document Collection

Let's create a small knowledge base containing information about Python, AI, and Machine Learning.

In [ ]:
documents = [
    "Python is widely used for automation, web development, and data science.",
    "Machine Learning enables computers to learn patterns from data.",
    "Deep Learning uses neural networks with multiple layers.",
    "Natural Language Processing helps computers understand human language.",
    "FAISS enables efficient similarity search across vector embeddings.",
    "Retrieval-Augmented Generation retrieves external information to support LLM responses.",
    "Vector embeddings represent the semantic meaning of text as numerical vectors.",
    "Semantic search retrieves relevant information based on meaning rather than keywords."
]

Each document will be converted into an embedding and indexed for retrieval.

---

## 🤖 Step 3 — Generate Document Embeddings

We'll use the pre-trained `all-MiniLM-L6-v2` model, which generates **384-dimensional embeddings**.

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(
    documents,
    convert_to_numpy=True
)

embeddings = np.asarray(embeddings, dtype="float32")

print("Embedding Shape:", embeddings.shape)

**Expected output:**

```
Embedding Shape: (8, 384)
```

Each document is represented by a numerical vector that captures aspects of its semantic meaning.

---

## 🗂️ Step 4 — Build the FAISS Index

FAISS provides several indexing methods. We'll use `IndexFlatIP` to perform exact similarity searches.

Before indexing, we'll normalise our embeddings so that inner-product similarity becomes equivalent to cosine similarity.

In [ ]:
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Indexed Documents:", index.ntotal)

**Output:**

```
Indexed Documents: 8
```

Our document embeddings are now stored in the FAISS index.

---

## 🔍 Step 5 — Perform Semantic Search

Suppose a user asks:

**"How do computers understand human language?"**

We'll convert this query into an embedding using the same model.

In [ ]:
query = "How do computers understand human language?"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True
)

query_embedding = np.asarray(query_embedding, dtype="float32")
faiss.normalize_L2(query_embedding)

Now, retrieve the three most relevant documents.

In [ ]:
top_k = 3

scores, indices = index.search(query_embedding, top_k)

FAISS returns the document indices and their corresponding similarity scores.

---

## 📊 Step 6 — Display Search Results

Let's display the retrieved documents in descending order of similarity.

In [ ]:
print("Search Query:", query)

for rank, (doc_index, score) in enumerate(zip(indices[0], scores[0]), start=1):
    print(f"\nRank {rank}")
    print(f"Document: {documents[doc_index]}")
    print(f"Similarity Score: {score:.4f}")

The document about **Natural Language Processing** should generally rank highly because it closely matches the meaning of the query.

The similarity score indicates how closely the query embedding aligns with each document embedding.

**Note:** Higher similarity scores indicate stronger semantic relationships, but they don't guarantee factual correctness or relevance in every situation.

---

## ⚙️ Step 7 — Create a Reusable Search Function

We can simplify the process by creating a reusable function that accepts any query.

In [ ]:
def semantic_search(query, top_k=3):
    query_vector = model.encode([query], convert_to_numpy=True)
    query_vector = np.asarray(query_vector, dtype="float32")
    faiss.normalize_L2(query_vector)

    scores, indices = index.search(query_vector, min(top_k, index.ntotal))

    return [
        {"document": documents[i], "score": float(score)}
        for i, score in zip(indices[0], scores[0])
    ]

Let's test another query.

In [ ]:
results = semantic_search("How does AI learn from data?")

for result in results:
    print(result)

The search engine will retrieve documents related to Machine Learning and other semantically relevant concepts.

---

## 🔗 How Semantic Search Fits into RAG

Semantic search is the retrieval component of a RAG pipeline.

```python
Documents → Embeddings → FAISS Index
                              ↑
User Query → Query Embedding → Similarity Search
                              ↓
                       Relevant Documents
                              ↓
                         LLM Response
```

The retrieved documents provide additional context that an LLM can use when generating its response.

FAISS performs similarity search; it doesn't generate answers itself.

---

## 🌍 Real-World Applications

Semantic search using FAISS can support:

- **Document Search:** Retrieve relevant information from PDFs and knowledge bases.
- **RAG Applications:** Find useful context before generating LLM responses.
- **Customer Support:** Retrieve troubleshooting instructions based on user questions.
- **Enterprise Search:** Search internal documentation using natural language.
- **Recommendation Systems:** Identify related content based on semantic similarity.

---

## 🧠 Key Takeaways

- Semantic search retrieves information based on meaning rather than exact keywords.
- Sentence Transformers converts text into numerical embeddings.
- FAISS indexes embeddings for efficient similarity search.
- Normalised vectors allow inner-product search to represent cosine similarity.
- The same embedding model should be used for documents and queries.
- Retrieved documents can provide context for RAG applications.

---

## Conclusion

Semantic Search is an important component of Retrieval-Augmented Generation systems because it enables applications to retrieve information based on meaning.

In this project, we generated document embeddings using Sentence Transformers, indexed them using FAISS, and retrieved relevant documents through cosine similarity.

We also created a reusable semantic search function that can support different queries.

---